# Tree of Thoughts Phase 2: Enhanced Algorithm Implementation & LLM Integration

## Overview

This notebook provides a comprehensive implementation of Tree of Thoughts (ToT) Phase 2 with:
- Multiple search strategies (DFS, BFS, Best-First/Beam Search)
- LLM integration patterns for thought generation and evaluation
- Cost tracking and performance analysis
- Real-world examples and practical workflows
- Visualization and benchmarking tools

**Key Concepts:**
- **Thought Generation**: Using LLMs to generate candidate next steps
- **Thought Evaluation**: Scoring thoughts to guide search
- **Search Strategies**: Exploring the tree efficiently
- **Cost-Benefit Trade-offs**: Token usage vs solution quality


# Section 1: Core Infrastructure Classes

We'll build the foundation for ToT with reusable components.

In [ ]:
import json
import math
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Callable, Any
from enum import Enum
import random
from collections import deque
import heapq
from datetime import datetime
import re

# For visualization
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import numpy as np
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False

print(f'Matplotlib available: {HAS_MATPLOTLIB}')
print('Core imports completed successfully.')

## 1.1: Search Strategy Enums and Metrics

In [ ]:
class SearchStrategy(Enum):
    DFS = 'depth_first'
    BFS = 'breadth_first'
    BEST_FIRST = 'best_first'

@dataclass
class SearchMetrics:
    strategy: SearchStrategy
    nodes_explored: int = 0
    llm_calls: int = 0
    estimated_tokens: int = 0
    solution_found: bool = False
    solution_quality: float = 0.0
    max_depth_reached: int = 0
    execution_time_ms: float = 0.0
    solution_path: List[str] = field(default_factory=list)
    
    def __str__(self):
        return (f'SearchMetrics({self.strategy.value}):\\n'
                f'  Nodes: {self.nodes_explored}\\n'
                f'  LLM Calls: {self.llm_calls}\\n'
                f'  Est. Tokens: {self.estimated_tokens}\\n'
                f'  Solution Found: {self.solution_found}\\n'
                f'  Quality: {self.solution_quality:.2f}\\n'
                f'  Max Depth: {self.max_depth_reached}\\n'
                f'  Time: {self.execution_time_ms:.1f}ms')

print('SearchStrategy and SearchMetrics classes defined.')

## 1.2: Prompt Templates

In [ ]:
@dataclass
class GenerationPromptTemplate:
    name: str
    system_prompt: str
    user_template: str
    num_candidates: int = 3
    temperature: float = 0.7
    
    def format(self, context: str) -> str:
        return self.user_template.format(context=context)

@dataclass
class EvaluationPromptTemplate:
    name: str
    system_prompt: str
    user_template: str
    scoring_scale: int = 10
    
    def format(self, thought: str, context: str) -> str:
        return self.user_template.format(thought=thought, context=context)

print('Prompt template classes defined.')

## 1.3: Core Data Structures

In [ ]:
@dataclass
class Thought:
    content: str
    parent_id: Optional[str] = None
    score: float = 0.0
    is_terminal: bool = False
    metadata: Dict[str, Any] = field(default_factory=dict)
    
    def __hash__(self):
        return hash(self.content)
    
    def __eq__(self, other):
        if not isinstance(other, Thought):
            return False
        return self.content == other.content

@dataclass
class TreeNode:
    node_id: str
    thought: Thought
    depth: int
    parent_id: Optional[str] = None
    children: List[str] = field(default_factory=list)
    visit_count: int = 0
    best_child_score: float = 0.0
    
    def __lt__(self, other):
        return self.thought.score > other.thought.score

print('Thought and TreeNode classes defined.')

## 1.4: Mock LLM Implementation

In [ ]:
class MockLLMCallerWithCostTracking:
    def __init__(self, seed: int = 42):
        self.seed = seed
        random.seed(seed)
        self.call_count = 0
        self.total_tokens = 0
    
    def estimate_tokens(self, text: str) -> int:
        return max(10, len(text) // 4)
    
    def generate_thoughts(self, context: str, num_candidates: int = 3,
                         domain: str = 'default', temperature: float = 0.7) -> List[Thought]:
        self.call_count += 1
        tokens = self.estimate_tokens(context) + 50
        self.total_tokens += tokens
        
        responses = [
            'Try grouping numbers to find intermediate results',
            'Look for common factors or products',
            'Explore using division and multiplication pairs',
            'Consider bracket combinations',
            'Try reordering operations to find patterns'
        ]
        
        selected = random.sample(responses, min(num_candidates, len(responses)))
        thoughts = [Thought(content=r) for r in selected]
        return thoughts
    
    def evaluate_thought(self, thought: str, context: str, rubric: Optional[str] = None) -> float:
        self.call_count += 1
        tokens = self.estimate_tokens(thought + context) + 30
        self.total_tokens += tokens
        
        score = 0.5
        if 10 < len(thought) < 200:
            score += 0.2
        
        action_words = ['try', 'check', 'verify', 'test', 'consider']
        if any(word in thought.lower() for word in action_words):
            score += 0.15
        
        return min(1.0, score)
    
    def reset(self):
        self.call_count = 0
        self.total_tokens = 0

print('MockLLMCallerWithCostTracking class defined.')

# Section 2: Search Strategies

## Depth-First Search Implementation

In [ ]:
class DepthFirstSearcher:
    def __init__(self, llm: MockLLMCallerWithCostTracking,
                 generation_template: GenerationPromptTemplate,
                 max_depth: int = 5, max_iterations: int = 100):
        self.llm = llm
        self.generation_template = generation_template
        self.max_depth = max_depth
        self.max_iterations = max_iterations
        self.tree: Dict[str, TreeNode] = {}
        self.node_counter = 0
        self.metrics = SearchMetrics(strategy=SearchStrategy.DFS)
    
    def _generate_node_id(self) -> str:
        node_id = f'node_{self.node_counter}'
        self.node_counter += 1
        return node_id
    
    def search(self, initial_thought: str, context: Dict):
        import time
        start_time = time.time()
        
        root_id = self._generate_node_id()
        root_thought = Thought(content=initial_thought)
        root_node = TreeNode(node_id=root_id, thought=root_thought, depth=0)
        self.tree[root_id] = root_node
        
        solution = self._dfs_recursive(root_id, context, [0])
        
        self.metrics.nodes_explored = len(self.tree)
        self.metrics.llm_calls = self.llm.call_count
        self.metrics.estimated_tokens = self.llm.total_tokens
        self.metrics.solution_found = solution is not None
        self.metrics.max_depth_reached = max(
            (n.depth for n in self.tree.values()), default=0)
        self.metrics.execution_time_ms = (time.time() - start_time) * 1000
        
        return solution, self.metrics
    
    def _dfs_recursive(self, node_id: str, context: Dict, iteration_count: List[int]):
        current_node = self.tree[node_id]
        iteration_count[0] += 1
        
        if iteration_count[0] > self.max_iterations or current_node.depth >= self.max_depth:
            return None
        
        candidates = self.llm.generate_thoughts(
            context=current_node.thought.content,
            num_candidates=self.generation_template.num_candidates,
            domain=context.get('domain', 'default')
        )
        
        for candidate in candidates:
            score = self.llm.evaluate_thought(candidate.content, current_node.thought.content)
            candidate.score = score
        
        candidates.sort(key=lambda t: t.score, reverse=True)
        
        for candidate in candidates:
            child_id = self._generate_node_id()
            child_node = TreeNode(
                node_id=child_id,
                thought=candidate,
                depth=current_node.depth + 1,
                parent_id=node_id
            )
            self.tree[child_id] = child_node
            current_node.children.append(child_id)
            
            solution = self._dfs_recursive(child_id, context, iteration_count)
            if solution is not None:
                return solution
        
        return None

print('DepthFirstSearcher class defined.')

## Breadth-First Search Implementation

In [ ]:
class BreadthFirstSearcher:
    def __init__(self, llm: MockLLMCallerWithCostTracking,
                 generation_template: GenerationPromptTemplate,
                 max_depth: int = 5, branching_factor: int = 3, score_threshold: float = 0.3):
        self.llm = llm
        self.generation_template = generation_template
        self.max_depth = max_depth
        self.branching_factor = branching_factor
        self.score_threshold = score_threshold
        self.tree: Dict[str, TreeNode] = {}
        self.node_counter = 0
        self.metrics = SearchMetrics(strategy=SearchStrategy.BFS)
    
    def _generate_node_id(self) -> str:
        node_id = f'node_{self.node_counter}'
        self.node_counter += 1
        return node_id
    
    def search(self, initial_thought: str, context: Dict):
        import time
        start_time = time.time()
        
        root_id = self._generate_node_id()
        root_thought = Thought(content=initial_thought)
        root_node = TreeNode(node_id=root_id, thought=root_thought, depth=0)
        self.tree[root_id] = root_node
        
        queue = deque([root_id])
        solution = None
        
        while queue and solution is None:
            node_id = queue.popleft()
            current_node = self.tree[node_id]
            
            if current_node.depth >= self.max_depth:
                continue
            
            candidates = self.llm.generate_thoughts(
                context=current_node.thought.content,
                num_candidates=self.generation_template.num_candidates,
                domain=context.get('domain', 'default')
            )
            
            for candidate in candidates:
                score = self.llm.evaluate_thought(candidate.content, current_node.thought.content)
                candidate.score = score
            
            candidates.sort(key=lambda t: t.score, reverse=True)
            top_candidates = candidates[:self.branching_factor]
            
            for candidate in top_candidates:
                if candidate.score >= self.score_threshold:
                    child_id = self._generate_node_id()
                    child_node = TreeNode(
                        node_id=child_id,
                        thought=candidate,
                        depth=current_node.depth + 1,
                        parent_id=node_id
                    )
                    self.tree[child_id] = child_node
                    current_node.children.append(child_id)
                    queue.append(child_id)
        
        self.metrics.nodes_explored = len(self.tree)
        self.metrics.llm_calls = self.llm.call_count
        self.metrics.estimated_tokens = self.llm.total_tokens
        self.metrics.solution_found = solution is not None
        self.metrics.max_depth_reached = max(
            (n.depth for n in self.tree.values()), default=0)
        self.metrics.execution_time_ms = (time.time() - start_time) * 1000
        
        return solution, self.metrics

print('BreadthFirstSearcher class defined.')

## Beam Search Implementation

In [ ]:
class BeamSearcher:
    def __init__(self, llm: MockLLMCallerWithCostTracking,
                 generation_template: GenerationPromptTemplate,
                 max_depth: int = 5, beam_width: int = 3):
        self.llm = llm
        self.generation_template = generation_template
        self.max_depth = max_depth
        self.beam_width = beam_width
        self.tree: Dict[str, TreeNode] = {}
        self.node_counter = 0
        self.metrics = SearchMetrics(strategy=SearchStrategy.BEST_FIRST)
    
    def _generate_node_id(self) -> str:
        node_id = f'node_{self.node_counter}'
        self.node_counter += 1
        return node_id
    
    def search(self, initial_thought: str, context: Dict):
        import time
        start_time = time.time()
        
        root_id = self._generate_node_id()
        root_thought = Thought(content=initial_thought)
        root_node = TreeNode(node_id=root_id, thought=root_thought, depth=0)
        self.tree[root_id] = root_node
        
        current_beam = [root_id]
        solution = None
        
        for depth in range(1, self.max_depth + 1):
            if not current_beam or solution is not None:
                break
            
            next_candidates = []
            
            for parent_id in current_beam:
                parent_node = self.tree[parent_id]
                
                candidates = self.llm.generate_thoughts(
                    context=parent_node.thought.content,
                    num_candidates=self.generation_template.num_candidates,
                    domain=context.get('domain', 'default')
                )
                
                for candidate in candidates:
                    score = self.llm.evaluate_thought(candidate.content, parent_node.thought.content)
                    candidate.score = score
                    
                    child_id = self._generate_node_id()
                    child_node = TreeNode(
                        node_id=child_id,
                        thought=candidate,
                        depth=depth,
                        parent_id=parent_id
                    )
                    self.tree[child_id] = child_node
                    parent_node.children.append(child_id)
                    next_candidates.append((candidate.score, child_id))
            
            if next_candidates:
                next_candidates.sort(key=lambda x: x[0], reverse=True)
                current_beam = [node_id for _, node_id in next_candidates[:self.beam_width]]
            else:
                break
        
        self.metrics.nodes_explored = len(self.tree)
        self.metrics.llm_calls = self.llm.call_count
        self.metrics.estimated_tokens = self.llm.total_tokens
        self.metrics.solution_found = solution is not None
        self.metrics.max_depth_reached = max(
            (n.depth for n in self.tree.values()), default=0)
        self.metrics.execution_time_ms = (time.time() - start_time) * 1000
        
        return solution, self.metrics

print('BeamSearcher class defined.')

# Section 3: Evaluators

In [ ]:
class HeuristicEvaluator:
    def __init__(self, domain: str = 'default'):
        self.domain = domain
        self.llm_calls = 0
    
    def evaluate(self, thought: str, context: str) -> float:
        score = 0.5
        if 8 < len(thought) < 300:
            score += 0.15
        
        actions = ['try', 'check', 'verify', 'test', 'consider', 'evaluate']
        if any(action in thought.lower() for action in actions):
            score += 0.1
        
        if self.domain == 'math':
            keywords = ['number', 'operation', 'factor', 'divide', 'multiply']
            if any(kw in thought.lower() for kw in keywords):
                score += 0.2
        
        return min(1.0, score)

class LLMEvaluator:
    def __init__(self, llm: MockLLMCallerWithCostTracking, domain: str = 'default'):
        self.llm = llm
        self.domain = domain
        self.llm_calls = 0
    
    def evaluate(self, thought: str, context: str) -> float:
        self.llm_calls += 1
        return self.llm.evaluate_thought(thought, context)

class HybridEvaluator:
    def __init__(self, llm: MockLLMCallerWithCostTracking, domain: str = 'default',
                 heuristic_threshold: float = 0.3):
        self.heuristic = HeuristicEvaluator(domain)
        self.llm_eval = LLMEvaluator(llm, domain)
        self.heuristic_threshold = heuristic_threshold
        self.llm_calls = 0
    
    def evaluate(self, thought: str, context: str) -> float:
        h_score = self.heuristic.evaluate(thought, context)
        if 0.3 < h_score < 0.7:
            self.llm_calls += 1
            return self.llm_eval.evaluate(thought, context)
        return h_score

print('Evaluator classes defined.')

# Section 4: Complete Searcher Interface

In [ ]:
class TreeOfThoughtsSearcher:
    def __init__(self, llm: MockLLMCallerWithCostTracking):
        self.llm = llm
        self.searchers = {}
    
    def configure_dfs(self, generation_template: GenerationPromptTemplate,
                     max_depth: int = 5, max_iterations: int = 100):
        self.searchers['dfs'] = DepthFirstSearcher(self.llm, generation_template, max_depth, max_iterations)
    
    def configure_bfs(self, generation_template: GenerationPromptTemplate,
                     max_depth: int = 5, branching_factor: int = 3):
        self.searchers['bfs'] = BreadthFirstSearcher(self.llm, generation_template, max_depth, branching_factor)
    
    def configure_beam(self, generation_template: GenerationPromptTemplate,
                      max_depth: int = 5, beam_width: int = 3):
        self.searchers['beam'] = BeamSearcher(self.llm, generation_template, max_depth, beam_width)
    
    def search(self, strategy: str, initial_thought: str, context: Dict):
        if strategy not in self.searchers:
            raise ValueError(f'Unknown strategy: {strategy}')
        return self.searchers[strategy].search(initial_thought, context)

print('TreeOfThoughtsSearcher class defined.')

# Section 5: Practical Examples

## Example 1: Math Problem Solving

In [ ]:
math_gen_template = GenerationPromptTemplate(
    name='math_generation',
    system_prompt='You are a math problem solver.',
    user_template='Problem: {context}\nGenerate next steps.',
    num_candidates=3,
    temperature=0.6
)

print('Testing DFS on math problem...')
llm = MockLLMCallerWithCostTracking(seed=42)
searcher = TreeOfThoughtsSearcher(llm)
searcher.configure_dfs(math_gen_template, max_depth=3, max_iterations=20)

solution, metrics = searcher.search(
    'dfs',
    'Numbers: 3, 8, 3, 8. Make 24.',
    {'domain': 'math'}
)

print('\nSearch Results:')
print(metrics)

## Example 2: Strategy Comparison

In [ ]:
print('Comparing all three strategies on same problem...\n')

results = {}

# DFS
llm_dfs = MockLLMCallerWithCostTracking(seed=999)
dfs = DepthFirstSearcher(llm_dfs, math_gen_template, max_depth=3)
dfs_solution, dfs_metrics = dfs.search('Numbers: 3, 8, 3, 8. Make 24.', {'domain': 'math'})
results['DFS'] = dfs_metrics

# BFS
llm_bfs = MockLLMCallerWithCostTracking(seed=999)
bfs = BreadthFirstSearcher(llm_bfs, math_gen_template, max_depth=3)
bfs_solution, bfs_metrics = bfs.search('Numbers: 3, 8, 3, 8. Make 24.', {'domain': 'math'})
results['BFS'] = bfs_metrics

# Beam
llm_beam = MockLLMCallerWithCostTracking(seed=999)
beam = BeamSearcher(llm_beam, math_gen_template, max_depth=3, beam_width=2)
beam_solution, beam_metrics = beam.search('Numbers: 3, 8, 3, 8. Make 24.', {'domain': 'math'})
results['Beam'] = beam_metrics

print(f"{'Strategy':<10} {'Nodes':>8} {'Calls':>8} {'Tokens':>8} {'Depth':>8}")
print('-' * 45)
for name, metrics in results.items():
    print(f"{name:<10} {metrics.nodes_explored:>8} {metrics.llm_calls:>8} {metrics.estimated_tokens:>8} {metrics.max_depth_reached:>8}")

# Section 6: Best Practices Summary

## Key Takeaways

1. **Strategy Selection**
   - DFS: Deep reasoning chains
   - BFS: Breadth exploration
   - Beam: Balanced efficiency

2. **Cost Optimization**
   - Use hybrid evaluators to save LLM calls
   - Tune beam width for efficiency
   - Set reasonable depth limits

3. **Prompt Engineering**
   - Clear system prompts
   - Domain-specific guidance
   - Explicit evaluation criteria

4. **Real-World Deployment**
   - Replace MockLLMCallerWithCostTracking with actual API
   - Implement robust error handling
   - Monitor token usage and costs
   - Cache repeated evaluations

## Conclusion

Tree of Thoughts is powerful for complex reasoning tasks. Choose your strategy based on:
- Problem complexity (depth needed)
- Budget constraints (token limits)
- Quality requirements
- Time constraints